# Basic Understanding of Neural Networks

A neural network is a computer system inspired by the human brain.

It learns to solve problems by looking at examples, like:
- Predicting house prices from features
- Recognizing digits
- Classifying cats vs. dogs
- Detecting objects (like YOLO!)

A basic neural network looks like this: `Input Layer` ➡️ `Hidden Layers` ➡️ `Output Layer`

Each layer contains neurons (also called nodes), which:
- Take input
- Multiply it by a weight
- Add a bias
- Pass it through an activation function

The weights are what the model learns


**How Neural Networks Learn**
    
Learning happens through a process called backpropagation:

1. Predict output (forward pass)
2. Compare prediction to true answer (loss function)
3. Adjust weights to reduce error (gradient descent)

Repeat this for many examples — and the model becomes smarter over time.

# Basic Idea of CNNs

A Convolutional Neural Network (CNN) is a type of neural network specially designed to process images. It’s what lets computers do things like:
- Detect objects (like YOLO does)
- Classify images (cat vs dog)
- Read handwriting (digit recognition)
- Power facial recognition, self-driving cars, etc.

**Why Normal Neural Networks Don’t Work Well with Images**

Let’s say you have a 100x100 RGB image.

That’s 100 × 100 × 3 = 30,000 pixels

A normal neural network would need 30,000 input neurons! 😱

- ❌ That’s slow
- ❌ Doesn’t capture image structure (like edges, shapes)
- ✅ That’s why we use CNNs

## Key Building Blocks of a CNN

**1️⃣ Convolution Layer**

This is the core idea of CNNs. Instead of connecting every pixel to every neuron, we:
- Use a small filter (like 3×3 or 5×5) to slide over the image
- Multiply pixel values with the filter weights
- Sum them up — this gives feature maps

📌 This detects patterns like:
- Vertical edges
- Horizontal lines
- Corners
- Texture

**Example:**

| Input Image Patch | 3×3 Filter  | Output |
| ----------------- | ----------- | ------ |
| `[100, 120, 130]` | Edge filter | `+120` |

This way, CNN focuses on local features instead of the whole image at once.


**2️⃣ Activation Function (ReLU)**

After convolution, we apply ReLU (Rectified Linear Unit): `ReLU(x) = max(0, x)`

This:
- Removes negative values
- Adds non-linearity (so the model can learn complex patterns)


**3️⃣ Pooling Layer (Downsampling)**

This layer shrinks the image while keeping important info.

🔹 Most common is Max Pooling:

- Looks at small windows (like 2×2)
- Takes the maximum value
- Reduces size by half

*4️⃣ Flatten + Fully Connected (Dense) Layers**
After several Conv + Pooling layers, we flatten the output (convert to 1D vector) and pass it to a regular neural network layer.

This makes predictions like:
- Is it a cat or dog?
- What objects are in the image?

**How CNN Works as a Whole**
1. Convolution layers extract features (edges, textures, shapes)
2. Pooling layers reduce size and focus on important parts
3. Dense layers make final decisions based on all learned features

## Why YOLO Uses CNNs?
YOLO uses CNNs to:
- Extract features from images
- Detect objects and their locations
- Predict bounding boxes and class labels

The YOLO architecture is a big CNN with:
- Convolution layers → detect patterns
- Detection heads → predict boxes + labels



## How YOLO Uses CNN to Detect Objects

### 🧱 1. Input Image
YOLO takes an image (e.g., 640×640) and passes it through a CNN. `Input → CNN → Features`

### 🧠 2. Backbone (Feature Extractor)
This is the main CNN part.
- Convolutional layers detect features like edges, corners, textures
- Output is a feature map – a compact version of the image with important patterns

Example:
Image 640×640 → becomes a feature map of 20×20×256 (compressed)

In YOLOv8, common backbones: `CSPDarknet`, `C2f`, `Conv`, `Spp`, etc.

### 🧠 3. Neck (Feature Aggregation)

YOLO adds more layers to:
- Combine features from different depths
- Use features from both big and small objects

🔸 This helps detect:
- Small objects (e.g., pen)
- Large objects (e.g., car)

This includes:
- FPN (Feature Pyramid Network)
- PAN (Path Aggregation Network)

### 🧠 4. Head (Detection Layer)

Now, YOLO predicts from the feature map:

For each grid cell:
- 1 or more bounding boxes
- Each box has:
    - x_center, y_center, width, height
    - Confidence score (how sure it is there's an object)
    - Class probabilities (is it a dog, car, etc.)

### 🧮 5. How Prediction Works
Let’s say we divide the image into a 13×13 grid.
For each grid cell, YOLO predicts:

- Multiple bounding boxes (e.g., 3)
- Each box predicts:
    - 4 box coordinates → [x, y, w, h]
    - 1 object confidence → P(object)
    - N class probabilities → for N classes

So for each box:
→ (x, y, w, h, confidence, class1_score, class2_score, ...)


### 🧠 6. How YOLO Knows Where the Object Is?
💡 It doesn’t look for one object at a time.

It looks at the entire image once (hence: You Only Look Once)
→ and predicts many boxes at once using the feature map.

Then it uses:
- IoU (Intersection over Union) to compare predictions with ground truth
- NMS (Non-Maximum Suppression) to remove overlapping boxes

In [ ]:
Input Image
     ↓
Convolutional Layers (Backbone) → extract features
     ↓
Neck (FPN/PAN) → combine features from all layers
     ↓
Head → predict bounding boxes + class labels
     ↓
Apply NMS → final predictions


## How YOLO Predicts Multiple Objects from One Image

Unlike classifiers (which say “this is a cat”), YOLO detects many objects at once, and for each one, it predicts:
- Where it is (bounding box)
- What it is (class label)
- How confident it is (score)

So the question is: **❓ How can YOLO make multiple predictions from just one forward pass of the image?**

`YOLO treats the image as a grid.`

**📦 Step-by-step:**
1. Input image (e.g., 416×416 or 640×640)
2. Divide into an S × S grid (e.g., 7×7 for YOLOv1, 20×20 or more in newer versions)
3. Each grid cell is responsible for:
    - Predicting 1 or more bounding boxes
    - Confidence score (objectness)
    - Class probabilities

**So with a 7×7 grid and 2 boxes per cell → you get `98 predictions`!**

| Grid Size | Boxes per Cell | Total Boxes Predicted |
| --------- | -------------- | --------------------- |
| 7×7       | 2              | 7×7×2 = 98            |


Each box predicts:
- 4 box coordinates: `[x, y, w, h]`
- 1 confidence score: `P(object)`
- C class probabilities: `P(class_i | object)` for each class

**`[center_x, center_y, width, height, confidence, class1_prob, class2_prob, ...]`**



#### How Predictions Are Made

Let’s say this is your image:

In [ ]:
-------------------------
| 🐶       🛵          🍎 |
-------------------------


- The image is divided into 7×7 grid.
- The 🐶 (dog) lies in cell (2,2)
- The 🛵 (bike) lies in cell (4,4)
- The 🍎 (apple) lies in cell (6,6)

**Each of these grid cells will predict:**
- A bounding box
- That object’s class
- Confidence of detection

| Traditional Detector      | YOLO                         |
| ------------------------- | ---------------------------- |
| Slide window over image   | One-shot detection           |
| One object per prediction | Multiple predictions at once |
| Slow and complex          | Fast and end-to-end          |


Let’s say YOLO outputs a tensor like: Output shape: `[S, S, B × (5 + C)]`

Where:
- S = grid size (e.g., 7)
- B = number of bounding boxes per cell (e.g., 2)
- C = number of classes (e.g., 3 for dog, car, apple)

Each box contains: `[ x, y, w, h, confidence, class1, class2, class3 ]`

→ Multiply and reshape: all predictions in one big array.

After YOLO makes all these predictions:

1. Thresholding: Remove predictions with low confidence
2. Non-Maximum Suppression (NMS): Keep best box for each object, remove overlaps
3. Final Output: Clean boxes with class labels and scores


In [ ]:
# example output:

Predictions:
[
  [dog, 0.93, [x1, y1, x2, y2]],
  [bicycle, 0.88, [x1, y1, x2, y2]],
  [apple, 0.79, [x1, y1, x2, y2]]
]

# Each row: [class label, confidence, bounding box]

**Summary**

| Step                   | Description                           |
| ---------------------- | ------------------------------------- |
| Image Grid             | Image is split into S×S cells         |
| Prediction per cell    | Each cell predicts multiple boxes     |
| Full prediction tensor | `[S, S, B × (5 + C)]`                 |
| Post-processing (NMS)  | Removes overlaps and keeps best boxes |
| Final output           | Clean boxes with classes and scores   |


# 🧠 What is YOLO (You Only Look Once)?
YOLO is an object detection algorithm that can detect multiple objects in an image and say:
- What the object is (e.g., dog, car, person)
- Where the object is (bounding box around the object)

So, if you give it this image:

![Image of street with people and cars]

YOLO will return:

In [ ]:
Dog → at (x1, y1, x2, y2) → with 92% confidence  
Person → at (x1, y1, x2, y2) → with 88% confidence  
Car → at (x1, y1, x2, y2) → with 95% confidence

## 🧾 YOLO vs Other Techniques
Before YOLO, object detection was done using methods like:
- R-CNN, Fast R-CNN, Faster R-CNN — slow and multi-step

YOLO is different:
- ✅ It does everything in one go: detect all objects in a single neural network pass
- ✅ It is fast and can work in real-time


**💡 Simple Analogy:**

Think of YOLO like a person looking at a photo once and saying: `"I see a dog in the top-left, a car in the center, and a person on the right."` Whereas older methods would look at one spot at a time and repeat many times to figure it out.

## 🔧 What YOLO Does Internally (Simplified)
YOLO takes an image and:

1. Divides the image into an `SxS` grid
2. Each grid cell predicts:
    - Whether an object is present in that cell
    - The class of the object (like dog, car)
    - The bounding box (coordinates: x, y, width, height)
3. YOLO then merges the predictions and removes duplicates (Non-Maximum Suppression)

## ✅ Use Cases of YOLO
- Self-driving cars 🚗 (detect pedestrians, traffic lights)
- Security cameras 👀 (detect people, intruders)
- Retail analytics 🛒 (track customers)
- Wildlife monitoring 🐯
- Sports analysis ⚽

## How YOLO Predicts Multiple Objects from One Image

Unlike classifiers (which say “this is a cat”), YOLO detects many objects at once, and for each one, it predicts:
- Where it is (bounding box)
- What it is (class label)
- How confident it is (score)

So the question is: ❓ How can YOLO make multiple predictions from just one forward pass of the image?

**Key Idea: Divide and Conquer (Grid System)**

YOLO treats the image as a grid.

**📦 Step-by-step:**
1. Input image (e.g., 416×416 or 640×640)
2. Divide into an S × S grid (e.g., 7×7 for YOLOv1, 20×20 or more in newer versions)
3. Each grid cell is responsible for:
    - Predicting 1 or more bounding boxes
    - Confidence score (objectness)
    - Class probabilities

**So with a 7×7 grid and 2 boxes per cell → you get 98 predictions!**

# Object Detection Basics

Object Detection is a type of computer vision task that allows you to: `🔎 Locate and classify multiple objects in a single image`.

**It answers:**
- What objects are in the image? (classification)
- Where are those objects? (bounding boxes)

**📸 Example:**
    
For this image:
![Example: Image with dog, cat, person]

The output would be:

In [ ]:
Dog → (100, 150, 200, 300), confidence: 0.92
Cat → (250, 180, 320, 290), confidence: 0.88
Person → (30, 50, 100, 200), confidence: 0.95

Each object has:
- A class label
- A bounding box (x1, y1, x2, y2)
- A confidence score

**🧱 Key Concepts in Object Detection**

Let’s go through each important concept you should understand:

## Bounding Boxes
- A Bounding Box is a rectangle that surrounds an object in an image.
- It tells the model: 👉 “This is where the object is.”

### 📐 Bounding Box Formats
There are two common ways to represent a bounding box:

####  1. (x_min, y_min, x_max, y_max)

- `x_min`, `y_min`: top-left corner of the box
- `x_max`, `y_max`: bottom-right corner

- ✅ Easy to draw a box using pixel coordinates
- ❌ Not normalized (depends on image size)

#### 2. YOLO Format: (x_center, y_center, width, height)

👉 Normalized to image dimensions (values between 0 and 1)

Each value represents a fraction of the full image:

In [ ]:
(x_center / image_width,
 y_center / image_height,
 width / image_width,
 height / image_height)


**📌 Why YOLO uses this format?**
- It's simpler for neural networks to predict
- Keeps output values between 0 and 1
- Helps in generalization

#### 📄 Example 1

**🎯 Given:**
Image size:
- Width = IMAGE_WIDTH
- Height = IMAGE_HEIGHT

**Bounding Box:**
- Top-left corner: (`x_min`, `y_min`)
- Bottom-right corner: (`x_max`, `y_max`)

**Formulas:**

| Value          | Formula                                   |
| -------------- | ----------------------------------------- |
| Width of box   | `box_width = x_max - x_min`               |
| Height of box  | `box_height = y_max - y_min`              |
| Center X       | `x_center = x_min + box_width / 2`        |
| Center Y       | `y_center = y_min + box_height / 2`       |
| YOLO x\_center | `x_center_norm = x_center / IMAGE_WIDTH`  |
| YOLO y\_center | `y_center_norm = y_center / IMAGE_HEIGHT` |
| YOLO width     | `width_norm = box_width / IMAGE_WIDTH`    |
| YOLO height    | `height_norm = box_height / IMAGE_HEIGHT` |


**🎯 Image size** = 800 × 600

**🎯 Bounding box:**
- Top-left corner → (200, 100)
- Bottom-right corner → (600, 400)

In [ ]:
# Calculate Box Width and Height

box_width = x_max - x_min = 600 - 200 = 400  
box_height = y_max - y_min = 400 - 100 = 300


In [ ]:
# Find Center Coordinates (in pixels)

x_center = x_min + box_width / 2 = 200 + 400 / 2 = 200 + 200 = 400  
y_center = y_min + box_height / 2 = 100 + 300 / 2 = 100 + 150 = 250

# So, the center of the box is at (400, 250) in pixel space.

In [ ]:
# Normalize to YOLO Format

# Now divide each value by the image dimensions:

x_center_norm = 400 / 800 = 0.5  
y_center_norm = 250 / 600 ≈ 0.4167  
width_norm = 400 / 800 = 0.5  
height_norm = 300 / 600 = 0.5

In [ ]:
# Assuming class ID = 0 (for example: 0 = "car"), the YOLO annotation is:

0 0.5 0.4167 0.5 0.5

#### 📄 Example 2

Suppose you have an image of size 640 x 480 pixels, and there's a car in a bounding box with:
- Top-left corner at (160, 120)
- Bottom-right corner at (480, 360)

So:
- Width = 480 - 160 = 320
- Height = 360 - 120 = 240
- Center = ((160+480)/2, (120+360)/2) = (320, 240)

In YOLO format, we convert to:

In [ ]:
x_center = 320 / 640 = 0.5
y_center = 240 / 480 = 0.5
width    = 320 / 640 = 0.5
height   = 240 / 480 = 0.5


**👉 Final YOLO annotation:**

In [ ]:
class_id x_center y_center width height
0         0.5      0.5       0.5   0.5

# Where class_id = 0 (maybe 0 = car)

### 🧮 How YOLO Predicts Bounding Boxes

Let’s say YOLO divides the image into a 7x7 grid.

Each cell:
- Predicts whether it contains the center of an object
- Predicts bounding box coordinates relative to the cell
- Predicts the confidence and class probability

So for a dog whose center lies in grid cell (3,4), only that cell predicts the box:

In [ ]:
[ x_offset, y_offset, width, height, confidence, class_probs... ]

# YOLO uses a sigmoid function to predict values between 0–1.

## Confidence Score
This tells how confident the model is that an object (any object) exists in a bounding box, and how good the box is.

**It is also called Objectness Score.**

A confidence score tells us how sure the model is that:
- There is an object in the bounding box
- That object is of a certain class (like dog, car, person)

So, it answers two questions:
- "Is there something here?"
- "What is it likely to be?"

YOLO calculates the confidence as: `Confidence Score = P(object) × IOU(pred_box, truth_box)`

Where:
- `P(object)` is the probability that any object exists in the box
- `IOU(pred_box, truth_box)` is how well the predicted box overlaps with the actual box (more on IOU later)

So, the confidence score includes **both existence and accuracy of the box**.

Let’s say YOLO looks at a region and predicts:
- Object exists with probability = 0.9
- IOU with ground truth = 0.8

So: `Confidence Score = 0.9 × 0.8 = 0.72`

**So, the model is saying:** `“I’m 90% sure there’s some object here, and the box matches ground truth 80% well, so I give it a 72% confidence.”`

**💡 This score is calculated before the model even decides what kind of object it is.**

### Why Confidence Score Matters

YOLO may predict hundreds of boxes, but not all are useful. You usually filter predictions by setting a threshold:

In [ ]:
if confidence_score > 0.5:
    keep the prediction
else:
    discard it


You can adjust the threshold depending on your needs:

- High confidence = fewer false positives
- Low confidence = more detections (but riskier)

### 📊 Confidence + Class Probabilities
Once YOLO is confident there is an object, it also predicts which class it is (e.g., dog, cat, car).

In YOLO, each prediction has:
- Bounding Box
- Objectness score (confidence)
- Class probabilities (e.g., 0.9 for "car", 0.1 for "bike")

Then YOLO combines them: `Final Score for class = Confidence × Class Probability`

#### Example:
Let’s say YOLO predicts:
- Objectness = 0.8

Class probabilities:
- Dog = 0.1
- Car = 0.7
- Person = 0.2

In [ ]:
Dog confidence = 0.8 × 0.1 = 0.08
Car confidence = 0.8 × 0.7 = 0.56 ✅
Person confidence = 0.8 × 0.2 = 0.16

# So YOLO picks "Car" with a confidence of 0.56.

In [ ]:
# You’ll often see boxes like:

[Dog] 0.91
[Person] 0.88
[Car] 0.73

# These are the confidence scores YOLO outputs along with the boxes and class labels.

**YOLO is 91% confident that the object is a `dog`, based on both the box confidence (that there's something there) and the class probability (that it's a dog).**

#### Confidence Threshold Tuning (Real-World Tip)
In code, you’ll often do:

In [ ]:
results = model(image)
results = results[results.confidence > 0.5]  # filter weak detections

# For strict detection: use 0.7 or 0.8
# For sensitive systems (like cancer detection): maybe allow >0.3

## Class Labels

A class label tells the model what the detected object is. So if YOLO detects a bounding box, the class label tells you: `“What object is inside this box?”`

### In YOLO, How are Class Labels Represented?
YOLO doesn’t use names directly — instead, it assigns each class a numeric ID.

For example, in the COCO dataset (common in YOLO training), class labels are stored like this:

| Class Name | Class ID |
| ---------- | -------- |
| person     | 0        |
| bicycle    | 1        |
| car        | 2        |
| dog        | 16       |
| cat        | 15       |


### How YOLO Predicts the Class
For each bounding box, YOLO outputs:
- Class probabilities for all possible classes (like a softmax)
- The class with the highest probability is chosen

Example class prediction from model: `[‘person’: 0.80, ‘dog’: 0.10, ‘cat’: 0.05, ‘car’: 0.05]`
So YOLO chooses person as the class for this box.

### How Class Labels Work in Custom Models
If you're training YOLO on your own dataset, you define your own classes.

For example, if your project is about detecting fruits, your `data.yaml` would contain:

names:
  0: apple
  1: banana
  2: orange

So if YOLO predicts `class_id = 1`, it means the object is a banana.

## IoU – Intersection over Union

IoU (Intersection over Union) is a metric that measures how much the predicted bounding box overlaps with the actual (ground truth) bounding box.

**It answers:** `“How good is this predicted box compared to the correct one?”`

**Formula:** `IoU= Area of Union/Area of Overlap`
- Overlap = Area where predicted and ground truth boxes intersect
- Union = Total area covered by both boxes combined


**Why Is IoU Important?**
    
IoU is used to:
- 📏 Measure how accurate a bounding box is
- 🧹 Help YOLO filter out bad or duplicate predictions
- 🧪 Evaluate models using metrics like mAP (mean Average Precision)

**Real-World Example**

Imagine two boxes:

- 🔵 Ground truth box (actual dog location)
- 🔴 Predicted box (YOLO's guess)

They overlap partially:

In [ ]:
+-------------------------+  
|     Ground Truth        |  
|    +------------+       |  
|    | Predicted  |       |  
|    |   Box      |       |  
|    +------------+       |  
+-------------------------+  


Let’s say:
- Overlap area = 3000 pixels²
- Union area = 5000 pixels²

Then: `IoU = 3000 / 5000 = 0.6`

**That means the prediction is 60% correct in terms of position and size.**

**👉 Common practice:** If `IoU ≥ 0.5`, we consider the prediction "correct".

### How YOLO Uses IoU
**During Training:**
- YOLO uses IoU to match predicted boxes to ground truth
- Helps model learn better localization

**During Inference:**
- IoU is used in Non-Maximum Suppression (NMS)
→ to remove duplicate boxes (we’ll cover NMS next)

## Non-Maximum Suppression (NMS)

When YOLO detects objects, it may predict multiple bounding boxes for the same object. All those boxes may have high confidence.

👉 So, NMS is a method to keep only the best box and suppress (remove) the rest.

That’s why it’s called Non-Maximum Suppression: `"Suppress everything except the box with the maximum score."`

### Why Does YOLO Predict Multiple Boxes?

YOLO scans the image using multiple anchor boxes and grid cells. So:

- Multiple parts of the network may say: “Hey, I found a car here!”
- Each gives slightly different coordinates with slightly different confidence scores.

Without NMS, you would get:

In [ ]:
[Car] 0.93 → Box A  
[Car] 0.89 → Box B  
[Car] 0.75 → Box C  

# All of them refer to the same car! We don’t want 3 boxes for 1 object.

### How NMS Works (Step-by-Step)
Let’s say we have multiple boxes predicting the same object.

**Step 1:** Sort boxes by confidence score (highest first)

In [ ]:
Box A – [car], score: 0.95  
Box B – [car], score: 0.85  
Box C – [car], score: 0.80  


**Step 2:** Keep the box with highest score (Box A)

Now compare Box A with all others.

**Step 3:** Remove boxes with high IoU overlap with Box A

Let’s say:
- Box B IoU with A = 0.65 → too high → remove
- Box C IoU with A = 0.3 → keep

📌 Threshold is usually `IoU > 0.5`

**Step 4:** Repeat with remaining boxes

After keeping Box A and removing Box B, now compare Box C with others.

👉 This continues until all duplicate boxes are removed.

In [ ]:
# Before NMS:
[dog] Box1 → score 0.9  
[dog] Box2 → score 0.85 (IoU 0.7 with Box1)  
[dog] Box3 → score 0.6  (IoU 0.3 with Box1)

# After NMS:
Keep Box1 (best)
Remove Box2 (IoU too high with Box1)
Keep Box3 (low overlap)


### YOLO Code Example
YOLO libraries (like Ultralytics) do NMS automatically. But here’s how it might look under the hood:

In [ ]:
from torchvision.ops import nms

# boxes: tensor of [x1, y1, x2, y2]
# scores: confidence scores
# iou_threshold: usually 0.5
keep = nms(boxes, scores, iou_threshold=0.5)


# Setting Up YOLOv8 and Running Detection

In [1]:
!pip install ultralytics

You should consider upgrading via the 'e:\notebook\notebook_env\scripts\python.exe -m pip install --upgrade pip' command.



  Using cached torchvision-0.22.1-cp39-cp39-win_amd64.whl (1.7 MB)
  Using cached torch-2.7.1-cp39-cp39-win_amd64.whl (216.0 MB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl (26 kB)
  Using cached filelock-3.18.0-py3-none-any.whl (16 kB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached fsspec-2025.5.1-py3-none-any.whl (199 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


In [2]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # 'n' means nano version (fast and light)
model.predict('t3.jpg', save=True)

# Save the result with bounding boxes on the image in a runs/detect/predict folder

100%|█████████████████████████████████████████████████████████████████████████████| 6.25M/6.25M [00:00<00:00, 11.6MB/s]



image 1/1 E:\Notebook\YOLO\t3.jpg: 448x640 3 persons, 1 horse, 2 cows, 114.7ms
Speed: 171.1ms preprocess, 114.7ms inference, 1.9ms postprocess per image at shape (1, 3, 448, 640)
Results saved to runs\detect\predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted p

In [3]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 model (nano version)
model = YOLO('yolov8n.pt')
# Run detection on an image
results = model.predict(source='t3.jpg', save=False, conf=0.5)

# Print detected objects
for r in results:
    print(r.names)             # Class labels
    print(r.boxes.cls)         # Class IDs
    print(r.boxes.conf)        # Confidence scores
    print(r.boxes.xyxy)        # Bounding boxes



image 1/1 E:\Notebook\YOLO\t3.jpg: 448x640 2 persons, 1 horse, 2 cows, 118.9ms
Speed: 3.4ms preprocess, 118.9ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chai

You can adjust:
- `conf=0.5` → confidence threshold
- Use `'yolov8s.pt'`, `'yolov8m.pt'`, `'yolov8l.pt'`, `'yolov8x.pt'` for larger models

In [ ]:
# To run YOLO on webcam (0 = default camera):
model.predict(source=0, show=True)


In [ ]:
# To run it on a video file:
model.predict(source='video.mp4', save=True)


# Understanding YOLOv8 Output Format 

Once you run object detection using YOLOv8 like this:

In [6]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.predict(source='t3.jpg')


image 1/1 E:\Notebook\YOLO\t3.jpg: 448x640 3 persons, 1 horse, 2 cows, 145.7ms
Speed: 10.4ms preprocess, 145.7ms inference, 3.7ms postprocess per image at shape (1, 3, 448, 640)


You get a results object — but what exactly is inside that?

Let’s break it down so you can fully understand every part of the prediction.


## What’s in results?

The `results` returned by YOLOv8 is a list of `ultralytics.engine.results.Results` objects, one for each image.

In [7]:
results = model.predict(source='t3.jpg')
result = results[0]  # single image result
result


image 1/1 E:\Notebook\YOLO\t3.jpg: 448x640 3 persons, 1 horse, 2 cows, 115.9ms
Speed: 2.1ms preprocess, 115.9ms inference, 1.1ms postprocess per image at shape (1, 3, 448, 640)


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant',

**`result.boxes` – All Bounding Boxes**

This contains the detected objects in the image. You can access:

| Attribute           | What It Means                                   |
| ------------------- | ----------------------------------------------- |
| `result.boxes.xyxy` | Bounding box coordinates in `[x1, y1, x2, y2]`  |
| `result.boxes.conf` | Confidence scores (final scores shown on image) |
| `result.boxes.cls`  | Class IDs (`0 = person`, `1 = bicycle`, etc.)   |
| `result.names`      | Dictionary mapping class ID to label name       |


In [8]:
for box in result.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0]  # top-left and bottom-right corners

    label = result.names[class_id]
    print(f"{label}: {confidence:.2f} → Box: ({x1:.0f}, {y1:.0f}), ({x2:.0f}, {y2:.0f})")


cow: 0.88 → Box: (54, 69), (82, 135)
cow: 0.77 → Box: (92, 54), (151, 143)
person: 0.65 → Box: (142, 21), (203, 163)
horse: 0.56 → Box: (185, 50), (228, 154)
person: 0.50 → Box: (142, 20), (203, 102)
person: 0.49 → Box: (189, 45), (210, 102)


<hr>

**`result.orig_img` – Original Image**

You can access the original input image if you want to do custom plotting:

In [9]:
img = result.orig_img
img

array([[[  0,   2,   2],
        [  0,   0,   0],
        [  6,   8,   8],
        ...,
        [188, 196, 209],
        [181, 189, 202],
        [194, 202, 215]],

       [[  1,   3,   3],
        [  2,   4,   4],
        [  2,   4,   4],
        ...,
        [181, 189, 202],
        [177, 185, 198],
        [187, 195, 208]],

       [[  0,   0,   0],
        [  0,   0,   0],
        [  0,   1,   1],
        ...,
        [195, 201, 212],
        [191, 197, 208],
        [197, 203, 214]],

       ...,

       [[ 35,  36,  32],
        [ 43,  44,  40],
        [ 46,  47,  43],
        ...,
        [ 48,  49,  45],
        [ 48,  49,  45],
        [ 47,  48,  44]],

       [[ 34,  35,  31],
        [ 40,  41,  37],
        [ 37,  38,  34],
        ...,
        [ 49,  50,  46],
        [ 49,  50,  46],
        [ 48,  49,  45]],

       [[ 43,  44,  40],
        [ 46,  47,  43],
        [ 36,  37,  33],
        ...,
        [ 45,  46,  42],
        [ 45,  46,  42],
        [ 45,  46,  42]]

**`result.masks` (if using segmentation)**

If you use YOLOv8-seg models, the `.masks` attribute gives you pixel-wise object masks. For detection (`YOLOv8n.pt`), this will be `None`.

In [10]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.predict(source='t3.jpg', save=True)

result = results[0]  # first image

for box in result.boxes:
    class_id = int(box.cls[0])
    conf = float(box.conf[0])
    x1, y1, x2, y2 = map(int, box.xyxy[0])  # convert to integers
    label = result.names[class_id]

    print(f"Detected {label} with {conf:.2f} confidence at [{x1}, {y1}, {x2}, {y2}]")



image 1/1 E:\Notebook\YOLO\t3.jpg: 448x640 3 persons, 1 horse, 2 cows, 95.8ms
Speed: 2.9ms preprocess, 95.8ms inference, 1.5ms postprocess per image at shape (1, 3, 448, 640)
Results saved to runs\detect\predict2
Detected cow with 0.88 confidence at [54, 69, 82, 135]
Detected cow with 0.77 confidence at [92, 53, 150, 143]
Detected person with 0.65 confidence at [141, 21, 202, 162]
Detected horse with 0.56 confidence at [184, 50, 228, 154]
Detected person with 0.50 confidence at [142, 20, 202, 101]
Detected person with 0.49 confidence at [189, 45, 209, 102]


# Train YOLOv8 on Custom Dataset

| Step | Task                                                |
| ---- | --------------------------------------------------- |
| 1️⃣  | Prepare/collect your images                         |
| 2️⃣  | Annotate the objects in the images (bounding boxes) |
| 3️⃣  | Organize the dataset in YOLO format                 |
| 4️⃣  | Create a `data.yaml` file                           |
| 5️⃣  | Train the model using Ultralytics                   |


## YOLO Dataset Folder Structures

### Structure A: Roboflow Style (More Common)

In [ ]:
dataset/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
├── data.yaml

# ✔️ This is default for Roboflow, and YOLOv8 supports it natively.

Your `data.yaml` file should point like this:

In [ ]:
# data.yaml:

path: dataset  # optional if using absolute paths
train: train/images
val: valid/images

names:
  0: apple
  1: banana


### Structure B: Ultralytics Style (Also valid)

In [ ]:
dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
├── data.yaml

# ✔️ Also fully supported by YOLOv8.

In this case, your `data.yaml` looks like:

In [ ]:
# data.yaml:

path: dataset
train: images/train
val: images/val

names:
  0: apple
  1: banana


| Folder Style          | Tool Produces It          | Supported in YOLOv8 | Recommended For                   |
| --------------------- | ------------------------- | ------------------- | --------------------------------- |
| **Roboflow Style**    | Roboflow export           | ✅ Yes               | Beginners, pre-annotated datasets |
| **Ultralytics Style** | Manual labeling / scripts | ✅ Yes               | Custom annotation, automation     |


In [ ]:
from ultralytics import YOLO

# Load pretrained model (transfer learning)
model = YOLO("yolov8n.pt")  # or 'yolov8s.pt' / 'm.pt' / 'l.pt'

# Train model
model.train(
    data="D:/YOLO-Projects/helmet-dataset/data.yaml",  # path to your yaml
    epochs=50,
    imgsz=640,
    batch=16,
)


You can also run it directly from terminal: `yolo task=detect mode=train model=yolov8n.pt data="D:/YOLO-Projects/helmet-dataset/data.yaml" epochs=50 imgsz=640`

After training finishes, YOLO saves the model weights in: `runs/detect/train/weights/best.pt`

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")  # trained model

# Run prediction
results = model.predict(source="test_image.jpg", save=True, conf=0.5)

# Display result path
print("Saved to:", results[0].save_dir)


In [ ]:
# Webcam
model.predict(source=0, show=True)

# On video
model.predict(source="video.mp4", save=True)


# Evaluating the YOLOv8 Model

| Metric            | What It Means                                                      |
| ----------------- | ------------------------------------------------------------------ |
| **Precision**     | % of predicted boxes that were correct                             |
| **Recall**        | % of actual objects that were detected                             |
| **mAP\@0.5**      | Mean Average Precision at IoU=0.5 (common benchmark)               |
| **mAP\@0.5:0.95** | Average mAP over IoU thresholds 0.5 to 0.95 (stricter, COCO style) |


## When Is Evaluation Done?
YOLOv8 automatically evaluates your model during training at the end of each epoch — using the validation set.

You’ll see results like:

In [ ]:
Metrics:
  precision: 0.87
  recall: 0.82
  mAP50: 0.91
  mAP50-95: 0.68


These are saved in: `runs/detect/train/results.csv` And visualized in plots: `runs/detect/train/results.png`

## 🎯 Precision = TP / (TP + FP)
- TP: True Positives (correct detections)
- FP: False Positives (wrong detections)

High precision = your model makes fewer mistakes

## 🔍 Recall = TP / (TP + FN)
- FN: False Negatives (missed objects)

High recall = your model catches most objects

## 🏆 mAP (mean Average Precision)
The most important metric in object detection.

- Measures how well the model ranks correct boxes across all classes
- mAP@0.5: IOU threshold = 0.5
- mAP@0.5:0.95: averaged over IoU = 0.5, 0.55, …, 0.95

`mAP closer to 1.0 = better!`



You can also generate per-class metrics:

In [ ]:
model.val(data="data.yaml")

This runs evaluation only and gives you a detailed breakdown.

## When Should You Be Satisfied?

| Metric        | OK   | Good  | Excellent |
| ------------- | ---- | ----- | --------- |
| Precision     | >0.6 | >0.8  | >0.9      |
| Recall        | >0.6 | >0.8  | >0.9      |
| mAP\@0.5      | >0.7 | >0.85 | >0.9      |
| mAP\@0.5:0.95 | >0.4 | >0.6  | >0.75     |

**⚠️ Tip:** mAP@0.5:0.95 is harder to achieve and more important in industry (like COCO competition)

# Improving YOLOv8 Model Performance

## 1️⃣ 🧹 Clean and Balanced Dataset
- Make sure your labels are accurate
- Each image should:
    - Have clear, visible objects
    - Have the correct number of annotations

- Avoid:
    - Tiny objects hard to detect
    - Extremely imbalanced classes (e.g., 1000 apples but only 5 bananas)

- 🔧 Use Roboflow to:

    - Visualize data distribution
    - Spot label problems
    - Auto-clean/resize

## 2️⃣ 📈 Data Augmentation
Augmentation increases your data diversity without collecting new data.

✅ YOLOv8 has built-in augmentations:

In [ ]:
model.train(
    data='data.yaml',
    epochs=100,
    imgsz=640,
    augment=True  # enabled by default
)


🔁 What YOLOv8 might apply:

- Flipping (horizontal)
- Color jitter
- Cropping, rotation
- Scaling and translation
- Mosaic and MixUp (advanced)

You can customize augmentations too in advanced configs.

## 3️⃣ 🛠️ Hyperparameter Tuning
YOLOv8 already uses smart defaults, but tuning can help.

Use built-in training with `lr0` (learning rate), `batch`, etc.

In [ ]:
model.train(
    data='data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    lr0=0.01,       # initial learning rate
    patience=20,    # early stopping patience
)

# 🔧 Tip: Try increasing epochs gradually (start with 50–100)

## 4️⃣ 🔥 Transfer Learning (Fine-Tuning Pretrained Model)
Instead of training from scratch (`yolov8n.yaml`), start from pretrained weights:

In [ ]:
model = YOLO('yolov8n.pt')  # or yolov8s.pt, yolov8m.pt, etc.
model.train(data='data.yaml', epochs=100, imgsz=640)


**Benefits:**
- Starts with knowledge of general objects (from COCO)
- Learns your custom classes faster
- Works great even with smaller datasets

| File               | Type       | Purpose                                                          |
| ------------------ | ---------- | ---------------------------------------------------------------- |
| **`yolov8n.pt`**   | ✅ Weights  | Pretrained model weights (for inference/training)                |
| **`yolov8n.yaml`** | 🛠️ Config | Model **architecture configuration** (for training from scratch) |


In [ ]:
# Train from Scratch with yolov8n.yaml

from ultralytics import YOLO

model = YOLO('yolov8n.yaml')  # # Training from scratch (no pretrained weights), using config only
model.train(data='data.yaml', epochs=100)


Here, YOLO will:

- Build the model architecture using the `.yaml`
- Initialize weights randomly
- Train everything from scratch



## 5️⃣ 🧠 Choose the Right Model Size
YOLOv8 has 5 model sizes:

| Model        | Size    | Speed    | Accuracy  | Use If…                         |
| ------------ | ------- | -------- | --------- | ------------------------------- |
| `yolov8n.pt` | Nano    | 🚀 Fast  | 👍 Decent | Mobile/Real-time needs          |
| `yolov8s.pt` | Small   | Fast     | Better    | Balanced tasks                  |
| `yolov8m.pt` | Medium  | Moderate | Good      | Mid-sized datasets              |
| `yolov8l.pt` | Large   | Slower   | Very Good | Accuracy-focused projects       |
| `yolov8x.pt` | X-Large | 🐢 Slow  | 🔥 Best   | High-resource, best performance |


## 6️⃣ 🔄 Train with More Diverse Data
The best way to improve performance is often: 🗂️ More Data + Better Labels

- Add difficult images (blur, low light, occlusion)
- Label edge cases and rare classes
- Include different backgrounds, sizes, angles

You can use:

- Google Images + LabelImg
- Roboflow datasets
- Synthetic data (generate with Stable Diffusion or Unity)

# Anchor-Free Detection in YOLOv8

In older YOLO versions (YOLOv1 to YOLOv5), each grid cell predicted bounding boxes based on preset sizes and shapes, called anchor boxes.

Example: Let’s say a grid cell has 3 anchor boxes: `[80×80]`, `[120×60]`, `[30×150]`

These were used as starting guesses to detect objects of different shapes (like cars, persons, bottles).

➡️ The model didn’t predict absolute size — it predicted adjustments to anchors:

`final_box = anchor_box + predicted_offset`


## Problems with Anchor-Based Detection
- Hard to tune anchors manually
- Not optimal for new datasets
- Increased complexity
- Slower and less flexible

## Anchor-Free Detection in YOLOv8
YOLOv8 says: Forget anchor boxes!

Instead, it predicts:
1. Object center point (i.e. where the object is located)
2. Distances from center to the four sides of the bounding box

This is often called the distance-to-box format.

### How It Works
For every pixel (or point) in the feature map:

The model asks:
- Is this the center of any object?

If yes:
- It predicts 4 values:
    - left, top, right, bottom distances from that center point
- And class + objectness

**📦 Example:**

In [ ]:
At point (50, 60):
- left: 20 pixels
- top: 30 pixels
- right: 40 pixels
- bottom: 35 pixels

=> Bounding box = [x1, y1, x2, y2] using that info


This way, the model learns the size and position of boxes directly, no anchors involved.

| Feature                  | Anchor-Based (YOLOv5) | Anchor-Free (YOLOv8)        |
| ------------------------ | --------------------- | --------------------------- |
| Uses predefined boxes    | ✅ Yes                 | ❌ No                        |
| Learns box size directly | ❌ No                  | ✅ Yes                       |
| Output format            | Offsets from anchors  | Distances from center point |
| Simpler implementation   | ❌ No                  | ✅ Yes                       |
| Faster and flexible      | ❌ Less                | ✅ More                      |
